In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate


In [3]:
pdf_object=PyPDFLoader("../Data/iot.pdf")
pdf=pdf_object.load()

In [4]:
splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=60)

In [5]:
splitted_data=splitter.split_documents(pdf)

In [6]:
embeddings=OllamaEmbeddings(model="qwen3-embedding:8b")

In [8]:
vector_store=Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings,
    persist_directory="../Data"
)

In [10]:
prompt=PromptTemplate.from_template("""
You are a helpful assistant which answers questions based on the context and
replies sorry i dont know , its out of context , if you dont know
Context : {context}
Query   : {query} 
""")

In [21]:
query="who is the pm of india"


In [12]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context=""
    for doc in data :
        context=context + doc.page_content

    return {"context":context,"query":query}  

In [15]:
llm=ChatGroq(model="openai/gpt-oss-20b")

In [16]:
rag_chain=get_context|prompt|llm

In [22]:
answer=rag_chain.invoke(query)

In [23]:
answer

AIMessage(content="Sorry, I don't know. It's out of context.", additional_kwargs={'reasoning_content': 'We need to answer the query: "who is the pm of india" based on context. The context is about Chip Boot Flow, IoT, a date, a name Vipul Singh Negi, Department of CS and Engineering, NIT Rourkela. There\'s no mention of Indian PM. The instruction: "You are a helpful assistant which answers questions based on the context and replies sorry i dont know , its out of context , if you dont know". So we should respond "sorry i dont know, it\'s out of context" because the context does not provide that info.'}, response_metadata={'token_usage': {'completion_tokens': 139, 'prompt_tokens': 170, 'total_tokens': 309, 'completion_time': 0.144852906, 'completion_tokens_details': {'reasoning_tokens': 119}, 'prompt_time': 0.008209405, 'prompt_tokens_details': None, 'queue_time': 0.209356063, 'total_time': 0.153062311}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a12402de73', 'service_